# T05 — Frozen train / validation / held-out split

**Authorized scope:** build, verify, and seal the frozen 70/15/15 split. This notebook does not fit or apply any preprocessing, does not train a model, and does not read a held-out label, feature value, rate, or distribution — held-out row identities are written to an immutable artifact and never displayed here.

Flow: **Verify predecessor → build split → verify disjointness/regeneration → summarize train/validation support → seal held-out → persist immutable evidence.**

In [1]:
from __future__ import annotations

import copy
import hashlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data.py').is_file():
            return candidate
    raise RuntimeError(f'Could not locate repository root above {start}')


WORKING_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = _find_repo_root(WORKING_DIRECTORY)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    DataContractError, PROCESSED_COLUMNS, SOURCE_ROW_ID,
    finalize_artifact_manifest, load_selector, materialize_pandas,
    open_processed_dataset, portable_repo_path, sha256_file, write_json_new, write_text_new,
)
from src.audit import AuditContractError, validate_pre_split_frame
from src.split import (
    HeldOutAccessError, SplitContractError, SplitDataset,
    assign_split, held_out_support_gate, membership_hash, support_summary,
    verify_deterministic_regeneration, verify_disjoint_and_complete,
)

NOTEBOOK_PATH = REPO_ROOT / 'notebooks' / 'internal' / 't05_frozen_split.ipynb'
T05_CONFIG_PATH = REPO_ROOT / 'configs' / 't05_split.json'
T03_CONFIG_PATH = REPO_ROOT / 'configs' / 't03_audit.json'
SELECTOR_PATH = REPO_ROOT / 'configs' / 'data_manifest.json'
STAGE = 't05_frozen_split'
POPULATION = 't03a_cleared_released_rows'


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def notebook_source_sha256(path):
    payload = json.loads(path.read_text(encoding='utf-8'))
    source_only = [{'cell_type': c['cell_type'], 'source': ''.join(c.get('source', []))} for c in payload['cells']]
    return hashlib.sha256(json.dumps(source_only, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

## 1. Verify the T03-A predecessor

**Question:** Has T03-A actually been accepted as `T03_PRE_SPLIT_COMPLETE`, and does its authorizing run still reconcile? T05 is explicitly authorized while T03 as a whole is still open (T03-B and T03-C come after this split exists) — but the pre-split integrity gates must have passed first.

In [2]:
t05_config = json.loads(T05_CONFIG_PATH.read_text(encoding='utf-8'))
t03_config = json.loads(T03_CONFIG_PATH.read_text(encoding='utf-8'))

required_state = t05_config['input']['predecessor']['required_lifecycle_state']
if t03_config['lifecycle_state'] != required_state:
    raise SplitContractError(
        f"T03-A is not {required_state} (found {t03_config['lifecycle_state']!r}); T05 cannot proceed."
    )

authorizing_run_id = t03_config['lifecycle_state_evidence']['authorizing_run_id']
authorizing_manifest_path = REPO_ROOT / t03_config['lifecycle_state_evidence']['authorizing_run_manifest']
authorizing_manifest = json.loads(authorizing_manifest_path.read_text(encoding='utf-8'))
if not str(authorizing_manifest['status']).startswith('COMPLETED'):
    raise SplitContractError('T03-A authorizing run is not a completed run')

for guard, expected in t05_config['scope_guards'].items():
    assert expected is False, f'Scope guard {guard} must be False at notebook start'

print(f'T03-A lifecycle state: {t03_config["lifecycle_state"]} (authorizing run {authorizing_run_id})')

T03-A lifecycle state: T03_PRE_SPLIT_COMPLETE (authorizing run t03a_audit_20260818T072125Z_409014)


## 2. Load the manifest-selected processed population

The same checksum-verified processed dataset T03-A audited — never a raw re-scan, never a different file discovered by name.

In [3]:
selector = load_selector(SELECTOR_PATH, REPO_ROOT)
dataset = open_processed_dataset(selector)  # fails closed unless the selector-pinned SHA-256 matches
processed_sha256 = selector.payload['processed_sha256']

frame = materialize_pandas(dataset, columns=PROCESSED_COLUMNS, row_limit=None)
if tuple(frame.columns) != PROCESSED_COLUMNS:
    raise AuditContractError('Processed columns changed during materialization')

# Re-run the pre-split hard gates here too: T05 must not silently trust a stale T03-A snapshot.
_ = validate_pre_split_frame(frame, require_zero_based_complete=True)
print(f'Loaded {len(frame):,} rows, processed_sha256={processed_sha256}')

Loaded 13,979,592 rows, processed_sha256=fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c91fd384fa0c0b54fd8


## 3. Immutable T05 run initialization

In [4]:
RUN_CREATED_AT = utc_now()
RUN_ID = datetime.now(timezone.utc).strftime('t05_split_%Y%m%dT%H%M%SZ_%f')
RUN_ROOT = REPO_ROOT / 'outputs' / 'runs' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=False)

try:
    git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
    git_dirty = bool(subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip())
except (OSError, subprocess.CalledProcessError):
    git_head, git_dirty = None, None

run_config = copy.deepcopy(t05_config)
run_config.update({
    'run_id': RUN_ID, 'created_at_utc': RUN_CREATED_AT, 'stage': STAGE,
    'notebook_source_sha256': notebook_source_sha256(NOTEBOOK_PATH),
    'src_split_sha256': sha256_file(REPO_ROOT / 'src' / 'split.py'),
    'processed_sha256': processed_sha256,
    'authorizing_t03a_run_id': authorizing_run_id,
    'git_head': git_head, 'git_dirty': git_dirty,
})
write_json_new(RUN_ROOT, 'audit/run_config.json', run_config)
print(f'Run: {RUN_ID}')

Run: t05_split_20260818T073132Z_534290


## 4. Build the frozen split

Seed 42, joint `(treatment, conversion)` stratification, 70/15/15. `T` and `Y` are read only to build strata and never enter the returned membership table.

In [5]:
membership = assign_split(frame)
counts = membership['split'].value_counts()
shares = counts / len(membership)
display_summary = pd.DataFrame({'rows': counts, 'share': shares}).loc[['train', 'validation', 'held_out']]
display_summary

,rows,share
split,,
train,9785714,0.70
validation,2096938,0.15
held_out,2096940,0.15


## 5. Verify disjointness, full row accounting, and deterministic regeneration

In [6]:
disjoint_evidence = verify_disjoint_and_complete(membership, frame[SOURCE_ROW_ID])
assert disjoint_evidence['complete'] is True
assert sum(disjoint_evidence['pairwise_overlaps'].values()) == 0

split_hash = membership_hash(membership)
verify_deterministic_regeneration(frame, split_hash)  # rebuilds from scratch and fails closed on mismatch

write_json_new(RUN_ROOT, 'audit/split_manifest.json', {
    'run_id': RUN_ID, 'seed': 42, 'train_fraction': 0.70, 'validation_fraction': 0.15,
    'heldout_fraction': 0.15, 'stratification': 'joint(treatment,conversion)',
    'membership_sha256': split_hash, 'disjointness_evidence': disjoint_evidence,
    'deterministic_regeneration_verified': True,
})
print(f'Membership hash: {split_hash}')
print('Disjoint and complete:', disjoint_evidence['complete'])
print('Counts:', disjoint_evidence['counts'])

Membership hash: cfc3789fdff58cd401aeb6d6e50afe0b26026ef8466ee8e9a7f9c822725f214c
Disjoint and complete: True
Counts: {'train': 9785714, 'validation': 2096938, 'held_out': 2096940}


## 6. Train / validation support summaries, and an opaque held-out gate

Train and validation `(T,Y)` support is reported in full. Held-out support is verified internally but only a `PASS`/`FAIL` status is exposed — no held-out counts, rates, or distributions.

In [7]:
train_support = support_summary(frame, membership, 'train')
validation_support = support_summary(frame, membership, 'validation')
heldout_gate_status = held_out_support_gate(frame, membership)

write_json_new(RUN_ROOT, 'audit/train_support.json', train_support)
write_json_new(RUN_ROOT, 'audit/validation_support.json', validation_support)
write_json_new(RUN_ROOT, 'audit/heldout_support_gate.json', {'run_id': RUN_ID, 'status': heldout_gate_status})

print('Train support:', train_support)
print('Validation support:', validation_support)
print('Held-out support gate (opaque):', heldout_gate_status)

Train support: {'split': 'train', 'n': 9785714, 'counts': {'T=0,Y=0': 1465012, 'T=0,Y=1': 2844, 'T=1,Y=0': 8292160, 'T=1,Y=1': 25698}}
Validation support: {'split': 'validation', 'n': 2096938, 'counts': {'T=0,Y=0': 313931, 'T=0,Y=1': 610, 'T=1,Y=0': 1776891, 'T=1,Y=1': 5506}}
Held-out support gate (opaque): PASS


## 7. Persist sealed membership and prove the held-out guard fails closed

The full membership table (all three labels, including `held_out`) is written as one immutable, opaque artifact — required so T05 itself is verifiable and so a future authorized T17 process can read it. Reading it back through `SplitDataset` demonstrates the access guard: train/validation identities are free; held-out identities raise `HeldOutAccessError` until a real T17 release marker exists, which it does not yet.

In [8]:
membership_payload = membership.to_csv(index=False, lineterminator='\n')
write_text_new(RUN_ROOT, 'audit/split_membership.csv', membership_payload)

dataset_view = SplitDataset(membership=membership)
assert len(dataset_view.train_ids()) == train_support['n']
assert len(dataset_view.validation_ids()) == validation_support['n']

# Negative-test proof, run live against the real split: no release marker exists yet.
nonexistent_marker = RUN_ROOT / 'audit' / 'heldout_release_authorization.json'
try:
    dataset_view.held_out_ids(release_manifest_path=nonexistent_marker)
    guard_failed_closed = False
except HeldOutAccessError:
    guard_failed_closed = True
assert guard_failed_closed, 'Held-out guard did not fail closed'

write_json_new(RUN_ROOT, 'audit/heldout_guard_proof.json', {
    'run_id': RUN_ID,
    'guard_failed_closed_without_release_marker': guard_failed_closed,
    'release_marker_checked_path': portable_repo_path(nonexistent_marker, REPO_ROOT),
    'release_marker_exists': nonexistent_marker.is_file(),
})
print('Held-out guard failed closed as required:', guard_failed_closed)

Held-out guard failed closed as required: True


## 8. Close the run

In [9]:
summary = {
    'run_id': RUN_ID, 'status': 'COMPLETED_T05_SPLIT_SEALED',
    'processed_sha256': processed_sha256, 'membership_sha256': split_hash,
    'row_count': int(len(membership)), 'disjoint_and_complete': disjoint_evidence['complete'],
    'deterministic_regeneration_verified': True,
    'heldout_support_gate': heldout_gate_status,
    'heldout_guard_failed_closed': guard_failed_closed,
    'held_out_labels_or_features_accessed': False,
    'preprocessing_fit_or_applied': False,
    'models_trained': False,
}
write_json_new(RUN_ROOT, 'audit/t05_summary.json', summary)

finalize_artifact_manifest(
    RUN_ROOT, run_id=RUN_ID, final_status=summary['status'], created_at_utc=utc_now(),
    stage=STAGE, population=POPULATION,
    external_artifacts=[
        {'path': selector.processed_path.name, 'role': 'manifest_selected_processed_derivative', 'sha256': processed_sha256, 'status': 'PASS'},
        {'path': 'notebooks/internal/t05_frozen_split.ipynb#sources', 'role': 'human_readable_protocol_source', 'sha256': notebook_source_sha256(NOTEBOOK_PATH), 'status': 'PASS'},
        {'path': 'src/split.py', 'role': 'reusable_t05_contract', 'sha256': sha256_file(REPO_ROOT / 'src' / 'split.py'), 'status': 'PASS'},
    ],
)
summary

{'run_id': 't05_split_20260818T073132Z_534290',
 'status': 'COMPLETED_T05_SPLIT_SEALED',
 'processed_sha256': 'fd2739bb074a50fa0b6796429fe3745e52f287e8770b2c91fd384fa0c0b54fd8',
 'membership_sha256': 'cfc3789fdff58cd401aeb6d6e50afe0b26026ef8466ee8e9a7f9c822725f214c',
 'row_count': 13979592,
 'disjoint_and_complete': True,
 'deterministic_regeneration_verified': True,
 'heldout_support_gate': 'PASS',
 'heldout_guard_failed_closed': True,
 'held_out_labels_or_features_accessed': False,
 'preprocessing_fit_or_applied': False,
 'models_trained': False}